## 📋 ملخص شامل والخطوات التالية

### ✅ ما تم إنجازه:

1. **استكشاف البيانات (Insights)**: تحليل توزيعات، ارتباطات، وقيم شاذة
2. **تجميع البيانات (Clustering)**: KMeans, DBSCAN, والتجميع الهرمي
3. **نظام التوصيات**: توصيات مبنية على البيانات لكل عنقود
4. **التنبؤات**: تصنيف (SLA) وانحدار (وقت التسليم)
5. **التفسير (SHAP)**: فهم أهمية الميزات
6. **الحفظ**: نماذج، بيانات، ونتائج

### 🎯 الخطوات التالية:

1. **نشر النموذج** - استخدم النموذج المحفوظ للتنبؤ على بيانات جديدة
2. **التحسينات** - قم بضبط المعاملات (Hyperparameter Tuning)
3. **المراقبة** - تابع أداء النموذج مع مرور الوقت
4. **التكامل** - ادمج مع نظام Mind-Q الرئيسي

---

In [ ]:
# ============================================
# 🔟 حفظ النماذج والنتائج
# ============================================

# إنشاء مجلد للحفظ
import os
output_dir = '/content/drive/MyDrive/Mind-Q/outputs'
os.makedirs(output_dir, exist_ok=True)

# 1. حفظ النماذج المدربة
if best_model is not None:
    model_name, trained_model, _, _ = best_model
    model_path = os.path.join(output_dir, f'best_model_{model_name.replace(\" \", \"_\")}.joblib')
    joblib.dump(trained_model, model_path)
    print(f\"✅ تم حفظ النموذج: {model_path}\")\n    
    # حفظ المقياس
    scaler_path = os.path.join(output_dir, 'scaler.joblib')
    joblib.dump(scaler_pred, scaler_path)
    print(f\"✅ تم حفظ المقياس: {scaler_path}\")\n\n# 2. حفظ الـ Clustering Models\nkmeans_path = os.path.join(output_dir, 'kmeans_model.joblib')
njoblib.dump(kmeans_final, kmeans_path)
nprint(f\"✅ تم حفظ KMeans: {kmeans_path}\")\n\n# 3. حفظ النتائج كـ JSON\noutputs = {\n    'insights': insights,\n    'recommendations': recommendations,\n    'cluster_profiles': {k: {**v, 'numeric_stats': {}} for k, v in cluster_profiles.items()},\n    'model_results': results if 'results' in locals() else {}\n}\n\nresults_path = os.path.join(output_dir, 'analysis_results.json')\nwith open(results_path, 'w', encoding='utf-8') as f:\n    json.dump(outputs, f, ensure_ascii=False, indent=2)\nprint(f\"✅ تم حفظ النتائج: {results_path}\")\n\n# 4. حفظ البيانات المعالجة\nprocessed_data_path = os.path.join(output_dir, 'processed_data.parquet')\ndf_processed.to_parquet(processed_data_path)\nprint(f\"✅ تم حفظ البيانات المعالجة: {processed_data_path}\")\n\n# 5. حفظ ملخص التقرير\nreport = f\"\"\"\\n# 📊 تقرير تحليل Mind-Q\n## ملخص النتائج\n\n### البيانات:\n- عدد الصفوف: {len(df)}\n- عدد الأعمدة: {len(df.columns)}\n- الأعمدة الرقمية: {len(df.select_dtypes(include=[np.number]).columns)}\n- الأعمدة الفئوية: {len(df.select_dtypes(include=['object']).columns)}\n\n### التجميع (Clustering):\n- أفضل عدد عناقيد (KMeans): {best_k}\n- عدد العناقيد (DBSCAN): {n_clusters_dbscan}\n- نقاط الضوضاء (Noise): {n_noise}\n\n### التنبؤ:\n- أفضل نموذج: {model_name if best_model else 'N/A'}\n- دقة النموذج: {best_score if best_model else 'N/A'}\n\n### التوصيات:\n- عدد التوصيات المولدة: {len(recommendations)}\n\"\"\"\n\nreport_path = os.path.join(output_dir, 'analysis_report.md')\nwith open(report_path, 'w', encoding='utf-8') as f:\n    f.write(report)\nprint(f\"✅ تم حفظ التقرير: {report_path}\")\n\nprint(f\"\\n✅ تم حفظ جميع النتائج في: {output_dir}\")"

## 💾 حفظ النماذج والـ Pipelines والنتائج

In [ ]:
# ============================================
# 9️⃣ تفسير النتائج (SHAP)
# ============================================

if best_model is not None:
    model_name, model, X_train_scaled, X_test_scaled = best_model
    
    print(f\"📊 تفسير النموذج الأفضل: {model_name}\\n\")\n    
    # SHAP Explainer
    explainer = shap.TreeExplainer(model) if hasattr(model, 'tree_') or 'Forest' in model_name or 'XGB' in model_name else shap.KernelExplainer(model.predict, X_train_scaled[:100])
    shap_values = explainer.shap_values(X_test_scaled[:100])
    
    # للـ Classification
    if hasattr(shap_values, '__len__') and isinstance(shap_values, list):
        shap_values = shap_values[1]  # استخدم قيم الفئة الموجبة
    
    # Summary Plot
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_test_scaled[:100], feature_names=X.columns, show=False, plot_type=\"bar\")
    plt.title(f\"SHAP Summary - أهم الميزات ({model_name})\")
    plt.tight_layout()
    plt.show()
    
    # Dependence Plot لأهم 3 ميزات
    if hasattr(shap_values, 'shape'):\n        feature_importance = np.abs(shap_values).mean(axis=0)
        top_features_idx = np.argsort(feature_importance)[-3:]
        
        fig = make_subplots(rows=1, cols=3, subplot_titles=[f\"Feature {X.columns[i]}\" for i in top_features_idx])
        
        for idx, feature_idx in enumerate(top_features_idx):
            fig.add_trace(go.Scatter(x=X_test_scaled[:100, feature_idx], 
                                     y=shap_values[:, feature_idx],
                                     mode='markers',
                                     name=X.columns[feature_idx]),
                         row=1, col=idx+1)
        
        fig.update_layout(height=400, title_text=\"تأثير أهم 3 ميزات\")
        fig.show()\n\nprint(\"✅ تم إنجاز تفسير النتائج!\")

## 📍 تفسير النتائج باستخدام SHAP

In [ ]:
# ============================================
# 8️⃣ التنبؤات (PREDICTIONS)
# ============================================

# إعداد البيانات للتنبؤ
X = df_processed.drop(columns=['order_id', 'KMeans_Cluster', 'DBSCAN_Cluster', 'Agglomerative_Cluster'] + 
                               [col for col in df_processed.columns if 'sla_achieved' in col.lower()])

# ======== مثال 1: التنبؤ بـ SLA Compliance (Classification) ========
if 'sla_achieved' in df.columns:
    y_sla = df['sla_achieved'].values
    
    X_train, X_test, y_train, y_test = train_test_split(X, y_sla, test_size=0.2, random_state=42)
    
    # تطبيع البيانات
    scaler_pred = StandardScaler()
    X_train_scaled = scaler_pred.fit_transform(X_train)
    X_test_scaled = scaler_pred.transform(X_test)
    
    # تدريب نماذج متعددة
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'XGBoost': XGBClassifier(n_estimators=100, random_state=42, verbosity=0)
    }
    
    results = {}
    best_model = None
    best_score = 0
    
    print(\"🤖 نتائج التنبؤ بـ SLA Compliance:\\n\")\n    for name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        
        results[name] = {
            'accuracy': accuracy,\n            'precision': precision,
            'recall': recall,
            'f1': f1
        }
        
        print(f\"{name}:\\n    Accuracy:  {accuracy:.3f}\\n    Precision: {precision:.3f}\\n    Recall:    {recall:.3f}\\n    F1-Score:  {f1:.3f}\\n\")\n        
        if accuracy > best_score:
            best_score = accuracy
            best_model = (name, model, X_train_scaled, X_test_scaled)
    
    # رسم النتائج
    results_df = pd.DataFrame(results).T
    fig = px.bar(results_df, barmode='group', title='مقارنة النماذج')\n    fig.show()

# ======== مثال 2: التنبؤ بوقت التسليم (Regression) ========
if 'delivery_time_hours' in df.columns:
    y_time = df['delivery_time_hours'].values
    
    X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X, y_time, test_size=0.2, random_state=42)
    
    X_train_reg_scaled = scaler_pred.fit_transform(X_train_reg)
    X_test_reg_scaled = scaler_pred.transform(X_test_reg)
    
    # نماذج الانحدار
    reg_models = {
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
        'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
    }
    
    print(\"\\n🎯 نتائج التنبؤ بوقت التسليم:\\n\")\n    for name, model in reg_models.items():
        model.fit(X_train_reg_scaled, y_train_reg)
        y_pred_reg = model.predict(X_test_reg_scaled)
        
        mse = mean_squared_error(y_test_reg, y_pred_reg)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test_reg, y_pred_reg)
        
        print(f\"{name}:\\n    MSE:  {mse:.3f}\\n    RMSE: {rmse:.3f}\\n    R²:   {r2:.3f}\\n\")\n\nprint(\"✅ اكتمل التنبؤ بنجاح!\")

## 🔮 التنبؤات (Predictions): Classification & Regression

In [ ]:
# ============================================
# 7️⃣ نظام التوصيات (RECOMMENDATION SYSTEM)
# ============================================

class RecommendationEngine:
    """محرك توصيات متقدم للبيانات اللوجستية"""
    
    def __init__(self, df, clusters):
        self.df = df.copy()
        self.clusters = clusters
        self.df['cluster'] = clusters
    
    def generate_recommendations(self):
        """توليد توصيات بناءً على الـ clusters والبيانات"""
        recommendations = []
        
        # تحليل كل عنقود
        for cluster_id in np.unique(self.clusters):
            cluster_data = self.df[self.df['cluster'] == cluster_id]
            
            # حساب الخصائص
            avg_delivery_time = cluster_data.get('delivery_time_hours', pd.Series()).mean()
            avg_rating = cluster_data.get('customer_rating', pd.Series()).mean()
            sla_rate = cluster_data.get('sla_achieved', pd.Series()).mean() if 'sla_achieved' in cluster_data else 0
            
            # توليد التوصيات
            if pd.notna(avg_delivery_time) and avg_delivery_time > 48:
                recommendations.append({
                    'cluster': cluster_id,
                    'type': 'Efficiency',
                    'action': 'تحسين وقت التسليم',
                    'reason': f'متوسط وقت التسليم {avg_delivery_time:.1f} ساعة (أكثر من 48)',
                    'priority': 'High'
                })
            
            if pd.notna(avg_rating) and avg_rating < 3.5:
                recommendations.append({
                    'cluster': cluster_id,
                    'type': 'Quality',
                    'action': 'تحسين جودة الخدمة',
                    'reason': f'متوسط التقييم {avg_rating:.1f}/5',
                    'priority': 'High'
                })
            
            if pd.notna(sla_rate) and sla_rate < 0.7:
                recommendations.append({
                    'cluster': cluster_id,
                    'type': 'SLA_Compliance',
                    'action': 'تحسين الامتثال للـ SLA',
                    'reason': f'معدل الامتثال {sla_rate*100:.1f}%',
                    'priority': 'Critical'
                })
        
        return recommendations
    
    def get_cluster_profiles(self):
        \"\"\"الحصول على ملفات تعريفية لكل عنقود\"\"\"
        profiles = {}
        
        for cluster_id in np.unique(self.clusters):
            cluster_data = self.df[self.df['cluster'] == cluster_id]
            
            profiles[f'Cluster_{cluster_id}'] = {
                'size': len(cluster_data),
                'percentage': f\"{len(cluster_data)/len(self.df)*100:.1f}%\",
                'numeric_stats': cluster_data.select_dtypes(include=[np.number]).describe().to_dict()
            }
        
        return profiles

# إنشاء محرك التوصيات
rec_engine = RecommendationEngine(df_processed, clusters_kmeans)
recommendations = rec_engine.generate_recommendations()
cluster_profiles = rec_engine.get_cluster_profiles()

# عرض التوصيات
print(\"🎯 التوصيات المولدة:\")\nfor idx, rec in enumerate(recommendations, 1):
    print(f\"\\n{idx}. {rec['action']}\")\n    - العنقود: {rec['cluster']}\n    - النوع: {rec['type']}\n    - السبب: {rec['reason']}\n    - الأولوية: {rec['priority']}\")\n\n# عرض ملفات تعريفية للعناقيد\nprint(\"\\n📊 ملفات تعريفية للعناقيد:\")\nfor cluster_name, profile in cluster_profiles.items():\n    print(f\"\\n{cluster_name}:\")\n    print(f\"  - الحجم: {profile['size']} ({profile['percentage']})\")

## 💬 نظام التوصيات (Recommendation System)

In [ ]:
# ============================================
# 6️⃣ التجميع (CLUSTERING)
# ============================================

# ======== KMeans ========
# 1. اختيار أفضل K باستخدام Elbow و Silhouette
inertias = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

# رسم Elbow و Silhouette
fig = make_subplots(rows=1, cols=2, subplot_titles=('Elbow Method', 'Silhouette Score'))

fig.add_trace(go.Scatter(x=list(K_range), y=inertias, mode='lines+markers',
                        name='Inertia'), row=1, col=1)

fig.add_trace(go.Scatter(x=list(K_range), y=silhouette_scores, mode='lines+markers',
                        name='Silhouette Score'), row=1, col=2)

fig.update_layout(height=400, title_text="اختيار أفضل عدد للعناقيد")
fig.show()

# اختيار أفضل K (من الـ Silhouette)
best_k = list(K_range)[np.argmax(silhouette_scores)]
print(f"✅ أفضل عدد عناقيد: {best_k}")

# 2. تطبيق KMeans بأفضل K
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
clusters_kmeans = kmeans_final.fit_predict(X_scaled)

# إضافة التسميات إلى البيانات الأصلية
df_processed['KMeans_Cluster'] = clusters_kmeans

print(f"✅ تم تطبيق KMeans مع {best_k} عناقيد")
print(f"   توزيع العناقيد: {np.bincount(clusters_kmeans)}")

# 3. تصور العناقيد على PCA
fig = px.scatter(x=X_pca_2d[:, 0], y=X_pca_2d[:, 1],
                color=clusters_kmeans, 
                title=f"KMeans Clustering (K={best_k}) على PCA",
                labels={'x': 'PC1', 'y': 'PC2'})
fig.show()

# ======== DBSCAN ========
from sklearn.neighbors import NearestNeighbors

# اختيار eps بناءً على k-distance graph
neighbors = NearestNeighbors(n_neighbors=5)
neighbors_fit = neighbors.fit(X_scaled)
distances, indices = neighbors_fit.kneighbors(X_scaled)
distances = np.sort(distances[:, 4], axis=0)

eps_optimal = distances[int(len(distances) * 0.95)]  # استخدم الـ 95th percentile
print(f"\n⚙️ DBSCAN - eps المقترح: {eps_optimal:.3f}")

dbscan = DBSCAN(eps=eps_optimal, min_samples=5)
clusters_dbscan = dbscan.fit_predict(X_scaled)

n_clusters_dbscan = len(set(clusters_dbscan)) - (1 if -1 in clusters_dbscan else 0)
n_noise = list(clusters_dbscan).count(-1)

print(f"✅ DBSCAN: {n_clusters_dbscan} عناقيد، {n_noise} نقطة noise")
df_processed['DBSCAN_Cluster'] = clusters_dbscan

# تصور DBSCAN
fig = px.scatter(x=X_pca_2d[:, 0], y=X_pca_2d[:, 1],
                color=clusters_dbscan.astype(str),
                title=f"DBSCAN Clustering على PCA",
                labels={'x': 'PC1', 'y': 'PC2'})
fig.show()

# ======== Agglomerative Clustering ========
agg_clustering = AgglomerativeClustering(n_clusters=best_k, linkage='ward')
clusters_agg = agg_clustering.fit_predict(X_scaled)

print(f"\n✅ Agglomerative Clustering: {best_k} عناقيد")
df_processed['Agglomerative_Cluster'] = clusters_agg

# تصور Agglomerative
fig = px.scatter(x=X_pca_2d[:, 0], y=X_pca_2d[:, 1],
                color=clusters_agg,
                title=f"Agglomerative Clustering (K={best_k}) على PCA",
                labels={'x': 'PC1', 'y': 'PC2'})
fig.show()

print("\n✅ اكتمل التجميع بنجاح!")

## 🎯 التجميع (Clustering): KMeans, DBSCAN, Hierarchical

In [ ]:
# ============================================
# 5️⃣ تقليل الأبعاد (Dimensionality Reduction)
# ============================================

# 1. تطبيع البيانات قبل PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_processed)

# 2. PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# حساب نسبة التباين المفسر التراكمي
cumsum = np.cumsum(pca.explained_variance_ratio_)

# رسم نسبة التباين
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(1, len(cumsum)+1)), y=cumsum, 
                        mode='lines+markers', name='Cumulative Variance'))
fig.add_hline(y=0.95, line_dash="dash", annotation_text="95% threshold")
fig.update_layout(title="PCA: نسبة التباين المفسر", 
                 xaxis_title="عدد المكونات",
                 yaxis_title="التباين المفسر التراكمي (%)")
fig.show()

print(f"📊 عدد المكونات اللازمة لـ 95% تباين: {np.argmax(cumsum >= 0.95) + 1}")

# 3. PCA بـ 2 أبعاد للتصور
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

# 4. t-SNE
print("\n⏳ جاري حساب t-SNE (قد يستغرق دقيقة أو دقيقتين)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_scaled)

# 5. UMAP (إذا كان متوفراً)
if UMAP_AVAILABLE:
    print("⏳ جاري حساب UMAP...")
    reducer = umap.UMAP(n_components=2, random_state=42)
    X_umap = reducer.fit_transform(X_scaled)
else:
    X_umap = None

# رسم النتائج
fig = make_subplots(rows=1, cols=3, subplot_titles=('PCA', 't-SNE', 'UMAP' if UMAP_AVAILABLE else 'N/A'))

fig.add_trace(go.Scatter(x=X_pca_2d[:, 0], y=X_pca_2d[:, 1], mode='markers', 
                        marker=dict(size=5, opacity=0.6), name='PCA'), row=1, col=1)

fig.add_trace(go.Scatter(x=X_tsne[:, 0], y=X_tsne[:, 1], mode='markers',
                        marker=dict(size=5, opacity=0.6), name='t-SNE'), row=1, col=2)

if UMAP_AVAILABLE:
    fig.add_trace(go.Scatter(x=X_umap[:, 0], y=X_umap[:, 1], mode='markers',
                            marker=dict(size=5, opacity=0.6), name='UMAP'), row=1, col=3)

fig.update_layout(height=400, title_text="تقليل الأبعاد (Dimensionality Reduction)", showlegend=False)
fig.show()

print("✅ تم إنجاز تقليل الأبعاد!")

## 📉 تقليل الأبعاد (Dimensionality Reduction): PCA, UMAP, t-SNE

In [ ]:
# ============================================
# 4️⃣ معالجة البيانات وتحضير الميزات
# ============================================

df_processed = df.copy()

# 1. التعامل مع القيم المفقودة
print("🔍 القيم المفقودة قبل المعالجة:")
print(df_processed.isnull().sum())

# ملء القيم المفقودة بالوسيط (للأعمدة الرقمية)
numeric_cols = df_processed.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df_processed[col].isnull().sum() > 0:
        df_processed[col].fillna(df_processed[col].median(), inplace=True)

# 2. ترميز المتغيرات الفئوية (One-Hot Encoding)
categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()
if categorical_cols:
    df_processed = pd.get_dummies(df_processed, columns=categorical_cols, drop_first=True)

print("\n✅ القيم المفقودة بعد المعالجة:")
print(df_processed.isnull().sum().sum())

print(f"\n📊 عدد الأعمدة بعد المعالجة: {df_processed.shape[1]}")
print(f"📊 عدد الصفوف: {df_processed.shape[0]}")

## 🧹 معالجة البيانات وتحضير الميزات (Feature Engineering)

In [ ]:
# رسوم بيانية Plotly تفاعلية
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# 1. توزيع الأعمدة الرقمية
fig = make_subplots(rows=2, cols=2, subplot_titles=numeric_cols[:4])
for idx, col in enumerate(numeric_cols[:4]):
    row = idx // 2 + 1
    col_pos = idx % 2 + 1
    fig.add_trace(go.Histogram(x=df[col], name=col, nbinsx=30), 
                  row=row, col=col_pos)
fig.update_layout(height=800, title_text="توزيع الأعمدة الرقمية", showlegend=False)
fig.show()

# 2. مصفوفة الارتباط
corr_matrix = df[numeric_cols].corr()
fig = go.Figure(data=go.Heatmap(z=corr_matrix.values, 
                                x=corr_matrix.columns, 
                                y=corr_matrix.columns,
                                colorscale='RdBu', zmid=0))
fig.update_layout(title="مصفوفة الارتباط (Correlation Matrix)", height=600, width=700)
fig.show()

# 3. توزيع الفئات
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
if categorical_cols:
    for col in categorical_cols[:3]:
        fig = px.bar(df[col].value_counts(), 
                     labels={'index': col, 'value': 'العدد'},
                     title=f"توزيع {col}")
        fig.show()

print("✅ تم عرض الرسوم البيانية!")

## 🎨 رسوم بيانية تفاعلية للـ Insights

In [ ]:
# ============================================
# 3️⃣ استكشاف البيانات والـ INSIGHTS
# ============================================

class InsightGenerator:
    """فئة لتوليد Insights من البيانات اللوجستية"""
    
    def __init__(self, df):
        self.df = df
        self.insights = {}
    
    def generate_all(self):
        """توليد جميع الـ Insights"""
        self.numeric_insights()
        self.categorical_insights()
        self.correlation_insights()
        self.anomaly_insights()
        return self.insights
    
    def numeric_insights(self):
        """Insights عن الأعمدة الرقمية"""
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        self.insights['numeric'] = {}
        
        for col in numeric_cols:
            data = self.df[col].dropna()
            self.insights['numeric'][col] = {
                'mean': data.mean(),
                'median': data.median(),
                'std': data.std(),
                'min': data.min(),
                'max': data.max(),
                'skewness': data.skew(),
                'kurtosis': data.kurtosis(),
                'q25': data.quantile(0.25),
                'q75': data.quantile(0.75),
            }
    
    def categorical_insights(self):
        """Insights عن الأعمدة الفئوية"""
        categorical_cols = self.df.select_dtypes(include=['object']).columns
        self.insights['categorical'] = {}
        
        for col in categorical_cols:
            value_counts = self.df[col].value_counts()
            self.insights['categorical'][col] = {
                'top_5': value_counts.head(5).to_dict(),
                'unique_count': self.df[col].nunique(),
                'mode': self.df[col].mode()[0] if len(self.df[col].mode()) > 0 else None,
            }
    
    def correlation_insights(self):
        """Insights عن الارتباطات"""
        numeric_df = self.df.select_dtypes(include=[np.number])
        corr_matrix = numeric_df.corr()
        
        # أعلى الارتباطات
        high_corr = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                if abs(corr_matrix.iloc[i, j]) > 0.5:
                    high_corr.append({
                        'feature1': corr_matrix.columns[i],
                        'feature2': corr_matrix.columns[j],
                        'correlation': corr_matrix.iloc[i, j]
                    })
        
        self.insights['correlations'] = high_corr
    
    def anomaly_insights(self):
        """الكشف عن القيم الشاذة (Outliers)"""
        numeric_df = self.df.select_dtypes(include=[np.number])
        self.insights['anomalies'] = {}
        
        for col in numeric_df.columns:
            Q1 = numeric_df[col].quantile(0.25)
            Q3 = numeric_df[col].quantile(0.75)
            IQR = Q3 - Q1
            outliers = ((numeric_df[col] < Q1 - 1.5*IQR) | (numeric_df[col] > Q3 + 1.5*IQR)).sum()
            self.insights['anomalies'][col] = {
                'outlier_count': int(outliers),
                'outlier_percentage': round(100 * outliers / len(numeric_df), 2)
            }

# إنشاء مولد الـ Insights
insight_gen = InsightGenerator(df)
insights = insight_gen.generate_all()

print("✅ تم توليد الـ Insights!")
print("\n🔍 أهم الارتباطات:")
for corr in insights['correlations'][:5]:
    print(f"  {corr['feature1']} <-> {corr['feature2']}: {corr['correlation']:.3f}")

print("\n🚨 القيم الشاذة (Anomalies):")
for col, anomaly in insights['anomalies'].items():
    if anomaly['outlier_count'] > 0:
        print(f"  {col}: {anomaly['outlier_count']} قيم شاذة ({anomaly['outlier_percentage']}%)")

## 💡 الخطوة 3: استكشاف البيانات والـ Insights

In [ ]:
# ============================================
# 2️⃣ تحميل البيانات
# ============================================

# الخيار 1: Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

# الخيار 2: قراءة من ملفات على Drive
# ضع المسار الصحيح لملفاتك
data_path = "/content/drive/MyDrive/Mind-Q/logistics_data.csv"  # غيّر المسار حسب احتياجك

# إذا كان الملف موجود، حمّله
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f"✅ تم تحميل {len(df)} صف من البيانات")
else:
    # بيانات تجريبية إذا لم يكن الملف موجوداً
    print("⚠️ الملف غير موجود، سنستخدم بيانات تجريبية")
    np.random.seed(42)
    n_samples = 1000
    
    df = pd.DataFrame({
        'order_id': range(1, n_samples + 1),
        'delivery_time_hours': np.random.exponential(24, n_samples),
        'distance_km': np.random.normal(50, 20, n_samples),
        'cod_amount': np.random.exponential(500, n_samples),
        'carrier': np.random.choice(['Carrier_A', 'Carrier_B', 'Carrier_C'], n_samples),
        'region': np.random.choice(['North', 'South', 'East', 'West'], n_samples),
        'sla_achieved': np.random.choice([0, 1], n_samples),
        'customer_rating': np.random.choice([1, 2, 3, 4, 5], n_samples),
        'num_attempts': np.random.randint(1, 5, n_samples),
        'weather': np.random.choice(['Sunny', 'Rainy', 'Cloudy'], n_samples),
    })

print("\n📊 ملخص البيانات:")
print(df.info())
print("\n📈 الإحصائيات الأساسية:")
print(df.describe())

## 📥 تحميل البيانات من Google Drive أو إرفاقها مباشرة

In [ ]:
# ============================================
# 1️⃣ استيراد المكتبات الضرورية
# ============================================

import numpy as np
import pandas as pd
import polars as pl
from scipy import stats
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score, calinski_harabasz_score
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier, XGBRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, confusion_matrix, classification_report,
                             mean_squared_error, r2_score)
from sklearn.pipeline import Pipeline
import joblib
import json
import warnings

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# SHAP for explainability
import shap

# Optional: UMAP for dimensionality reduction
try:
    import umap
    UMAP_AVAILABLE = True
except:
    print("⚠️ UMAP not available. Install with: pip install umap-learn")
    UMAP_AVAILABLE = False

# Set style and random seed
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')

print("✅ جميع المكتبات جاهزة!")
print(f"✅ Pandas version: {pd.__version__}")
print(f"✅ Polars version: {pl.__version__}")
print(f"✅ Scikit-learn version: {pd.__version__}")

# 🚀 Mind-Q Advanced Analytics Pipeline for Google Colab
## **Insights | Clustering | Recommendations | Predictions**

هذا النوتبوك يحول عمليات KNIME المعقدة إلى **كود Python عملي وسريع** على Google Colab.

### المحتويات:
1. ✅ استيراد المكتبات وتهيئة البيئة
2. ✅ تحميل البيانات (CSV, Parquet, Google Drive)
3. ✅ استكشاف البيانات والـ **Insights**
4. ✅ معالجة الميزات والتنظيف
5. ✅ **تقليل الأبعاد** (PCA, UMAP, t-SNE)
6. ✅ **Clustering** (KMeans, DBSCAN, Hierarchical)
7. ✅ **نظام التوصية** (Collaborative + Content-based)
8. ✅ **التنبؤ** (Classification & Regression)
9. ✅ **تفسير النتائج** (SHAP)
10. ✅ **حفظ النماذج والـ Pipelines**

---